In [9]:
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

session = requests.Session()

# ==============================================
# STEP 1 - Ambil homepage
# ==============================================

home_url = "https://21cineplex.com"
home_response = session.get(home_url)
home_html = home_response.text
home_soup = BeautifulSoup(home_html, "html.parser")

# ==============================================
# STEP 2 - Ambil token AJAX
# ==============================================

# token = re.search(r'token:\s*"([^"]+)"', home_html).group(1)

# print("Token ditemukan")

# # ==============================================
# # STEP 3 - Ambil semua kota
# # ==============================================

# city_dropdown = home_soup.find("select", id="cityChanged")

# cities = []

# for city in city_dropdown.find_all("option"):

#     cities.append({
#         "city_id": city["value"],
#         "city_name": city.text.strip()
#     })

# cities_df = pd.DataFrame(cities)

# print(f"Total kota: {len(cities_df)}")

# # ==============================================
# # STEP 4 - Loop semua kota
# # ==============================================

# all_movies = []

# for city in cities:

#     print(f"Processing: {city['city_name']}")

#     payload = {
#         "data": "getMovNowplaying",
#         "city_id": city["city_id"],
#         "token": token
#     }

#     response = session.post(
#         "https://21cineplex.com/script/ajax-home-nowplaying.php",
#         data=payload,
#         headers={
#             "Referer": "https://21cineplex.com/",
#             "X-Requested-With": "XMLHttpRequest"
#         }
#     )

#     try:
#         ajax_data = response.json()

#     except Exception:

#         print(f"Gagal di {city['city_name']}")

#         continue

#     ajax_soup = BeautifulSoup(ajax_data["msg"], "html.parser")

#     movie_cards = ajax_soup.find_all("div", class_="movie")

#     for movie in movie_cards:

#         try:

#             title = movie.find("h4").text.strip()

#             link = movie.find("a")["href"]

#             all_movies.append({
#                 "city_id": city["city_id"],
#                 "city_name": city["city_name"],
#                 "title": title,
#                 "link": link
#             })

#         except Exception:
#             pass

#     time.sleep(
#         random.uniform(0.5, 1.5)
#     )

# # ==============================================
# # STEP 5 - Hasil mentah
# # ==============================================

# movies_df = pd.DataFrame(all_movies)

# print(movies_df.shape)

In [5]:
import re

url = "https://21cineplex.com"
response = session.get(home_url)
# home_html = home_response.text
# home_soup = BeautifulSoup(home_html, "html.parser")

# 1. Pastikan response-nya sukses (Status 200)
print(f"Status Code: {response.status_code}")

# 2. Simpan hasil pencarian
match = re.search(r'token:\s*"([^"]+)"', home_html)

# 3. Cek pakai if
if match:
    token = match.group(1)
    print("✅ Token berhasil ditemukan!")
else:
    print("❌ Token GAGAL ditemukan!")
    
    # --- CARA MELIHAT MASALAHNYA ---
    # Cari kata 'token' di HTML, kalau ada, cetak 50 huruf sebelum & sesudahnya
    index_token = home_html.find('token')
    
    if index_token != -1:
        print("\nBentuk HTML aslinya ternyata seperti ini:")
        # Mencetak potongan HTML di sekitar kata 'token'
        print(home_html[index_token-20 : index_token+100]) 
    else:
        print("\nKata 'token' sama sekali tidak ada di HTML.")
        print("Kemungkinan kamu diblokir / dapat halaman error. Ini isi awal HTML-nya:")
        print(home_html[:300]) # Print 300 karakter pertama

Status Code: 403
❌ Token GAGAL ditemukan!

Kata 'token' sama sekali tidak ada di HTML.
Kemungkinan kamu diblokir / dapat halaman error. Ini isi awal HTML-nya:
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal


In [7]:
import re
import time
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup


# FUNGSI 1: ambil potongan teks di sekitar kata yang dicari
def potong_sekitar(teks, kata, maks=2):
    hasil = []
    teks_kecil = teks.lower()
    posisi = 0

    while len(hasil) < maks:
        idx = teks_kecil.find(kata.lower(), posisi)
        if idx == -1:
            break
        hasil.append(teks[max(0, idx - 100): idx + 400])
        posisi = idx + len(kata)

    return hasil


# FUNGSI 2: ambil file .js dari website yang sama
def ambil_file_js(html, base_url, session, maks_file=10):
    soup = BeautifulSoup(html, "html.parser")
    domain = urlparse(base_url).netloc
    isi_js = {}

    for tag in soup.find_all("script", src=True):
        link_js = urljoin(base_url, tag["src"])

        # lewati JS dari website lain (google, cdn, dll)
        if urlparse(link_js).netloc != domain:
            continue

        if len(isi_js) >= maks_file:
            break

        try:
            isi_js[link_js] = session.get(link_js).text
        except Exception as e:
            print(f"Gagal ambil {link_js}: {e}")

        time.sleep(1)

    return isi_js


# FUNGSI 3: pemeriksa utama
def inspect_website(html, base_url=None, session=None):

    # 1. kumpulkan semua teks yang mau diperiksa
    sumber = {"HTML halaman": html}

    if base_url and session:
        sumber.update(ambil_file_js(html, base_url, session))

    # 2. daftar pola yang dicari
    daftar_pola = {
        "jQuery AJAX": ["$.post", "$.get", "$.ajax"],
        "Axios": ["axios."],
        "Fetch API": ["fetch("],
        "XMLHttpRequest": ["XMLHttpRequest"],
        "GraphQL": ["graphql"],
        "Token / CSRF": ["csrf", "token:"],
        "Captcha": ["recaptcha", "hcaptcha", "turnstile"],
    }

    # 3. cari semua pola di semua sumber (tidak berhenti di temuan pertama)
    print("=== POLA YANG DITEMUKAN ===")
    ada_temuan = False

    for jenis, kata_kata in daftar_pola.items():
        for nama_sumber, teks in sumber.items():
            for kata in kata_kata:
                potongan = potong_sekitar(teks, kata, maks=1)
                if potongan:
                    ada_temuan = True
                    print(f"\n🟢 {jenis} (kata: {kata}) di {nama_sumber}")
                    print(potongan[0])

    if not ada_temuan:
        print("🔴 Tidak ada pola jelas. Lanjut ke DevTools Network.")

    # 4. cari alamat endpoint (.php, /api/, /ajax)
    endpoint = set()

    for teks in sumber.values():
        for hasil in re.findall(r'["\']([^"\']+\.php[^"\']*)["\']', teks):
            endpoint.add(hasil)
        for hasil in re.findall(r'["\']([^"\']*/(?:api|ajax)[^"\']*)["\']', teks):
            endpoint.add(hasil)

    print("\n=== KANDIDAT ENDPOINT ===")
    for alamat in sorted(endpoint):
        print(alamat)

    # 5. cek form (bisa jadi form login) dan input tersembunyi
    soup = BeautifulSoup(html, "html.parser")

    print("\n=== FORM ===")
    for form in soup.find_all("form"):
        nama_input = []
        for i in form.find_all("input"):
            if i.get("name"):
                nama_input.append(i.get("name"))
        print("action:", form.get("action"), "| method:", form.get("method"))
        print("input :", nama_input)

    return endpoint
#Cara pakainya, di sel terpisah:
endpoint = inspect_website(home_html, "https://21cineplex.com", session)

=== POLA YANG DITEMUKAN ===
🔴 Tidak ada pola jelas. Lanjut ke DevTools Network.

=== KANDIDAT ENDPOINT ===

=== FORM ===


In [10]:
def inspect_website(html):

    if "$.post" in html or "$.get" in html:

        print("🟢 jQuery AJAX terdeteksi")
        print("👉 Cari $.post() atau $.get()")

        idx = html.find("$.post")

        if idx == -1:
            idx = html.find("$.get")

        print(html[idx:idx+3000])

        return

    if "graphql" in html.lower():

        print("🟢 GraphQL terdeteksi")
        print("👉 Cari endpoint /graphql")

        idx = html.lower().find("graphql")

        print(html[idx-500:idx+3000])

        return

    if "axios" in html.lower():

        print("🟢 Axios terdeteksi")
        print("👉 Cari axios.get() atau axios.post()")

        idx = html.lower().find("axios")

        print(html[idx:idx+3000])

        return

    if "fetch(" in html:

        print("🟢 Fetch API terdeteksi")
        print("👉 Cari fetch()")

        idx = html.find("fetch(")

        print(html[idx:idx+3000])

        return

    if "/api/" in html or ".json" in html:

        print("🟢 REST API Candidate")
        print("👉 Cari endpoint /api/")

        idx = html.find("/api/")

        if idx != -1:
            print(html[idx-500:idx+1000])

        return

    print("🔴 Tidak ditemukan pola yang jelas")
    print("👉 Lanjut investigasi di DevTools Network")

In [11]:
inspect_website(home_html)

🔴 Tidak ditemukan pola yang jelas
👉 Lanjut investigasi di DevTools Network


In [14]:
import re
# GANTI IMPORT INI:
from curl_cffi import requests 

# 1. Bikin session, dan impersonate Chrome
session = requests.Session(impersonate="chrome")

# 2. Tembak halaman utama
response = session.get('https://21cineplex.com')
print(f"Status Code: {response.status_code}") # Pastikan 200, bukan 403

# 3. Ambil token (pakai home_html yang baru)
home_html = response.text
home_soup = BeautifulSoup(home_html,"html.parser")
match = re.search(r'token:\s*"([^"]+)"', home_html)

if match:
    token = match.group(1)
    print("✅ Token berhasil didapat!")
    print(f"Token: {token[:15]}...") # Print sebagian aja
else:
    print("❌ Masih gagal dapet token. Coba cek lagi.")

Status Code: 200
✅ Token berhasil didapat!
Token: 67d74772c38d0e6...


In [20]:
# ==============================================
# STEP 3 - Ambil semua kota
# ==============================================

city_dropdown = home_soup.find("select", id="cityChanged")

cities = []

for city in city_dropdown.find_all("option"):

    cities.append({
        "city_id": city["value"],
        "city_name": city.text.strip()
    })

cities_df = pd.DataFrame(cities)

print(f"Total kota: {len(cities_df)}")

# ==============================================
# STEP 4 - Loop semua kota
# ==============================================

all_movies = []

for city in cities:

    print(f"Processing: {city['city_name']}")

    payload = {
        "data": "getMovNowplaying",
        "city_id": city["city_id"],
        "token": token
    }

    response = session.post(
        "https://21cineplex.com/script/ajax-home-nowplaying.php",
        data=payload,
        headers={
            "Referer": "https://21cineplex.com/",
            "X-Requested-With": "XMLHttpRequest"
        }
    )

    try:
        ajax_data = response.json()

    except Exception:

        print(f"Gagal di {city['city_name']}")

        continue

    ajax_soup = BeautifulSoup(ajax_data["msg"], "html.parser")

    movie_cards = ajax_soup.find_all("div", class_="movie")

    for movie in movie_cards:

        try:

            title = movie.find("h4").text.strip()

            link = movie.find("a")["href"]

            all_movies.append({
                "city_id": city["city_id"],
                "city_name": city["city_name"],
                "title": title,
                "link": link
            })

        except Exception:
            pass

    time.sleep(
        random.uniform(0.5, 1.5)
    )

# ==============================================
# STEP 5 - Hasil mentah
# ==============================================

movies_df = pd.DataFrame(all_movies)

print(movies_df.shape)

Total kota: 71
Processing: AMBON
Processing: BALIKPAPAN
Processing: BANDUNG
Processing: BANJARMASIN
Processing: BATAM
Processing: BEKASI
Processing: BENGKULU
Processing: BOGOR
Processing: BONDOWOSO
Processing: BONTANG
Processing: CIANJUR
Processing: CIKARANG
Processing: CILEGON
Processing: CIREBON
Processing: DENPASAR
Processing: DEPOK
Processing: DUMAI
Processing: GARUT
Processing: GORONTALO
Processing: GRESIK
Processing: INDRAMAYU
Processing: JAKARTA
Processing: JAMBI
Processing: JAYAPURA
Processing: JEMBER
Processing: KARAWANG
Processing: KEDIRI
Processing: KENDARI
Processing: KISARAN
Processing: KLATEN
Processing: KUALA KAPUAS
Processing: KUPANG
Processing: LABUAN BAJO
Processing: LAMPUNG
Processing: MADIUN
Processing: MAGELANG
Processing: MAKASSAR
Processing: MALANG
Processing: MAMUJU
Processing: MANADO
Processing: MANOKWARI
Processing: MAROS
Processing: MATARAM
Processing: MEDAN
Processing: PADANG
Processing: PALANGKARAYA
Processing: PALEMBANG
Processing: PALU
Processing: PANGKAL

In [21]:
len(all_movies)

736

In [18]:
df = pd.DataFrame(all_movies)

In [19]:
df

,city_id,city_name,title,link
0,77,AMBON,How to Rob a Bank,"https://21cineplex.com/how-to-rob-a-bank,13904..."
1,77,AMBON,Hamka: Awal Mula (Vol 3),https://21cineplex.com/hamka-awal-mula-(vol-3)...
2,77,AMBON,Ramayana - Director's Cut,"https://21cineplex.com/ramayanadirectors-cut,1..."
3,77,AMBON,Pulang Kampung,"https://21cineplex.com/pulang-kampung,139027,1..."
4,77,AMBON,People Pleaser: Si Paling Gak Enakan,https://21cineplex.com/people-pleaser-si-palin...
...,...,...,...,...
770,75,YOGYAKARTA,Hungry Ghost Festival,"https://21cineplex.com/hungry-ghost-festival,1..."
771,75,YOGYAKARTA,Godzilla Minus Zero,"https://21cineplex.com/godzilla-minus-zero,138..."
772,75,YOGYAKARTA,The Hunger Games: Sunrise on the Reaping,https://21cineplex.com/the-hunger-games-sunris...
773,75,YOGYAKARTA,Laut Bercerita,"https://21cineplex.com/laut-bercerita,138241,1..."
